# Cryptogram Digits - EasyOCR Custom Model Training

## Dataset
- **Location**: `all_data/` (na raiz do projeto)
- **Images**: 27,180 (21,744 train + 5,436 val)

## Instructions
1. Set runtime to GPU: Runtime > Change runtime type > GPU (T4)
2. Upload `all_data.zip` to your Google Drive root (MyDrive/)
3. Run all cells sequentially
4. Training takes ~30-60 minutes

In [ ]:
!pip install easyocr torch torchvision pillow opencv-python lmdb pyyaml pandas
!git clone https://github.com/JaidedAI/EasyOCR.git
!mkdir -p all_data/train all_data/val saved_models
print("Setup complete!")

In [ ]:
from google.colab import drive
import os, shutil

# Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Path to all_data.zip in your Google Drive
# Update this path if your file is in a different location
drive_zip_path = '/content/drive/MyDrive/all_data.zip'

if not os.path.exists(drive_zip_path):
    print(f"ERROR: {drive_zip_path} not found!")
    print("Please upload all_data.zip to your Google Drive root (MyDrive/)")
else:
    print(f"Found: {drive_zip_path}")
    print("Extracting to all_data/...")
    !unzip -o {drive_zip_path} -d /content/
    
    train_count = len([f for f in os.listdir('all_data/train') if f.endswith('.png')])
    val_count = len([f for f in os.listdir('all_data/val') if f.endswith('.png')])
    print(f"Data ready: {train_count} train, {val_count} val")

In [ ]:
!wget -O saved_models/english_g2.zip https://github.com/JaidedAI/EasyOCR/releases/download/v1.3/english_g2.zip
!unzip -o saved_models/english_g2.zip -d saved_models/
import os
for f in os.listdir('saved_models/'):
    if f.endswith('.pth'):
        print(f"Model: saved_models/{f}")

In [ ]:
%%writefile train_digits.py
import os, sys, time, random, string
import torch, torch.nn as nn, torch.optim as optim, torch.utils.data
import numpy as np
from PIL import Image
import yaml

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class AttrDict(dict):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.__dict__ = self

class CTCLabelConverter:
    def __init__(self, character):
        self.character = character
        self.dict = {char: i + 1 for i, char in enumerate(character)}
        self.dict['[blank]'] = 0
    def encode(self, text, batch_max_length=25):
        length = [len(s) for s in text]
        encoded = torch.full((len(text), batch_max_length + 1), 0, dtype=torch.long)
        batch = torch.zeros(len(text), dtype=torch.long)
        for i, (t, l) in enumerate(zip(text, length)):
            batch[i] = l
            encoded[i][:l] = torch.LongTensor([self.dict[c] for c in t])
        return encoded, batch
    def decode(self, text_index, length):
        texts = []
        for idx, l in enumerate(length):
            t = text_index[idx, :l]
            # CTC greedy decode: remove blanks and duplicates
            chars = []
            prev = -1
            for i in t:
                if i != 0 and i != prev:  # skip blank (0) and duplicates
                    chars.append(self.character[i - 1])
                prev = i
            texts.append(''.join(chars))
        return texts

class Averager:
    def __init__(self): self.reset()
    def reset(self): self.n = 0; self.val = 0
    def add(self, val): self.n += 1; self.val = self.val - self.val/self.n + val/self.n

class Dataset(torch.utils.data.Dataset):
    def __init__(self, samples, opt):
        self.samples = samples
        self.opt = opt
    def __len__(self): return len(self.samples)
    def __getitem__(self, index):
        img_path, label = self.samples[index]
        try:
            img = Image.open(img_path).convert('L')
            img = img.resize((self.opt.imgW, self.opt.imgH), Image.BICUBIC)
            img = torch.FloatTensor(np.array(img) / 255.0).unsqueeze(0)
            return (img, label)
        except:
            return self.__getitem__(random.randint(0, len(self.samples) - 1))

class AlignCollate:
    def __init__(self, imgH=64, imgW=200):
        self.imgH, self.imgW = imgH, imgW
    def __call__(self, batch):
        images, labels = zip(*batch)
        return torch.stack(images, 0), labels

class VGG_FeatureExtractor(nn.Module):
    def __init__(self, input_channel, output_channel=256):
        super().__init__()
        oc = [output_channel//8, output_channel//4, output_channel//2, output_channel]
        self.ConvNet = nn.Sequential(
            nn.Conv2d(input_channel, oc[0], 3, 1, 1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(oc[0], oc[1], 3, 1, 1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(oc[1], oc[2], 3, 1, 1), nn.ReLU(True),
            nn.Conv2d(oc[2], oc[2], 3, 1, 1), nn.ReLU(True), nn.MaxPool2d((2,1),(2,1)),
            nn.Conv2d(oc[2], oc[3], 3, 1, 1, bias=False), nn.BatchNorm2d(oc[3]), nn.ReLU(True),
            nn.Conv2d(oc[3], oc[3], 3, 1, 1, bias=False), nn.BatchNorm2d(oc[3]), nn.ReLU(True),
            nn.MaxPool2d((2,1),(2,1)),
            nn.Conv2d(oc[3], oc[3], 2, 1, 0), nn.ReLU(True))
    def forward(self, x): return self.ConvNet(x)

class BiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.LSTM(input_size, hidden_size, bidirectional=True, batch_first=True)
        self.linear = nn.Linear(hidden_size * 2, output_size)
    def forward(self, x):
        if x.dim() != 3: x = x.unsqueeze(2)
        return self.linear(self.rnn(x)[0])

class Model(nn.Module):
    def __init__(self, opt):
        super().__init__()
        self.FeatureExtraction = VGG_FeatureExtractor(opt.input_channel, opt.output_channel)
        self.AdaptiveAvgPool = nn.AdaptiveAvgPool2d((1, None))
        self.SequenceModeling = nn.Sequential(
            BiLSTM(opt.output_channel, opt.hidden_size, opt.hidden_size),
            BiLSTM(opt.hidden_size, opt.hidden_size, opt.hidden_size))
        self.Prediction = nn.Linear(opt.hidden_size, opt.num_class)
    def forward(self, x, text):
        v = self.AdaptiveAvgPool(self.FeatureExtraction(x)).squeeze(2).permute(2, 0, 1)
        return self.Prediction(self.SequenceModeling(v).contiguous())

def load_dataset(root, opt):
    samples = []
    for dp, _, fns in os.walk(root):
        if 'gt.txt' in fns:
            with open(os.path.join(dp, 'gt.txt')) as f:
                for line in f:
                    line = line.strip()
                    if not line: continue
                    parts = line.split(',', 1)
                    if len(parts) == 2:
                        fname, label = parts[0], str(parts[1])
                        p = os.path.join(dp, fname)
                        if os.path.exists(p) and all(c in opt.character for c in label) and len(label) <= opt.batch_max_length:
                            samples.append((p, label))
    return Dataset(samples, opt)

def validate(model, criterion, loader, converter, opt, iteration=0):
    model.eval()
    total_loss, correct, n = 0, 0, 0
    with torch.no_grad():
        for batch_idx, (imgs, labels) in enumerate(loader):
            imgs = imgs.to(device)
            bs = imgs.size(0)
            text, length = converter.encode(labels, opt.batch_max_length)
            preds = model(imgs, text)
            preds_size = torch.IntTensor([preds.size(0)] * bs)
            total_loss += criterion(preds.log_softmax(2).cpu(), text, preds_size, length).mean().item() * bs
            _, idx = preds.max(2)
            preds_str = converter.decode(idx.cpu(), preds_size)
            correct += sum(p == l for p, l in zip(preds_str, labels))
            n += bs
            if iteration > 0 and batch_idx == 0 and iteration % 2000 == 0:
                print(f"\n  [DEBUG @ iter {iteration}] Predictions vs Ground Truth:")
                for p, l in zip(preds_str[:10], labels[:10]):
                    status = 'OK' if p == l else 'FAIL'
                    print(f"    Pred: '{p}' | True: '{l}' | {status}")
                print()
    return total_loss/n, correct/n

def main():
    with open('config_files/cryptogram_digits_config.yaml') as f:
        opt = AttrDict(yaml.safe_load(f))
    opt.character = opt.number + opt.symbol
    os.makedirs(f'./saved_models/{opt.experiment_name}', exist_ok=True)
    
    torch.backends.cudnn.benchmark = True
    random.seed(opt.manualSeed)
    np.random.seed(opt.manualSeed)
    torch.manual_seed(opt.manualSeed)

    print(f"\n{'='*60}")
    print(f"Training: {opt.experiment_name}")
    print(f"Characters: {opt.character}")
    print(f"Iterations: {opt.num_iter}, Batch: {opt.batch_size}, LR: {opt.lr}")
    print(f"Device: {device}")
    print(f"{'='*60}\n")

    collate = AlignCollate(opt.imgH, opt.imgW)
    train_ds = load_dataset(opt.train_data, opt)
    val_ds = load_dataset(opt.valid_data, opt)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=opt.batch_size, shuffle=True, num_workers=0, collate_fn=collate)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=min(32, opt.batch_size), shuffle=True, num_workers=0, collate_fn=collate)
    print(f"Train: {len(train_ds)}, Val: {len(val_ds)}\n")

    # Dataset distribution analysis
    label_counts = {}
    for _, label in train_ds.samples:
        label_counts[label] = label_counts.get(label, 0) + 1
    print("Dataset distribution (train):")
    for label in sorted(label_counts.keys()):
        print(f"  '{label}': {label_counts[label]} samples")
    print()

    converter = CTCLabelConverter(opt.character)
    opt.num_class = len(converter.character) + 1
    model = Model(opt).to(device)
    
    if opt.saved_model and os.path.exists(opt.saved_model):
        print(f"Loading pretrained: {opt.saved_model}")
        pretrained_state = torch.load(opt.saved_model, map_location=device)
        model_state = model.state_dict()
        
        # Strip 'module.' prefix if present
        new_state = {}
        for k, v in pretrained_state.items():
            name = k.replace('module.', '') if k.startswith('module.') else k
            new_state[name] = v
        pretrained_state = new_state
        
        # Remove Prediction layer (shape mismatch: 97 vs 11 classes)
        for key in list(pretrained_state.keys()):
            if 'Prediction' in key:
                del pretrained_state[key]
        
        matched = []
        skipped = []
        for k, v in pretrained_state.items():
            if k in model_state and v.shape == model_state[k].shape:
                matched.append(k)
            else:
                skipped.append(f"{k} (shape: {v.shape} vs {model_state.get(k, 'N/A').shape if k in model_state else 'MISSING'})")
        
        skipped.append("Prediction.* (intentionally removed - shape mismatch)")
        
        model.load_state_dict(pretrained_state, strict=False)
        print(f"  Matched: {len(matched)} layers")
        print(f"  Skipped: {len(skipped)} layers")
        if skipped:
            print(f"  Skipped layers (randomly initialized):")
            for s in skipped[:5]:
                print(f"    - {s}")
            if len(skipped) > 5:
                print(f"    ... and {len(skipped)-5} more")
        print()

    criterion = torch.nn.CTCLoss(zero_infinity=True)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=opt.lr, betas=(opt.beta1, 0.999))
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=opt.lr, total_steps=opt.num_iter, pct_start=0.1)

    loss_avg, best_acc, i = Averager(), 0.0, 0
    train_iter = iter(train_loader)

    print("\n[INFO] Training from scratch (no pretrained model)")
    print(f"[INFO] LR Schedule: OneCycleLR with warmup (10% of training)")
    print(f"[INFO] Initial LR: {opt.lr:.6f}, Max LR: {opt.lr:.6f}, Final LR: ~{opt.lr*0.01:.6f}\n")

    while i < opt.num_iter:
        try: imgs, labels = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            imgs, labels = next(train_iter)

        imgs = imgs.to(device)
        text, length = converter.encode(labels, opt.batch_max_length)
        preds = model(imgs, text)
        preds_size = torch.IntTensor([preds.size(0)] * imgs.size(0))
        cost = criterion(preds.log_softmax(2).cpu(), text, preds_size, length)
        optimizer.zero_grad()
        cost.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), opt.grad_clip)
        optimizer.step()
        scheduler.step()
        loss_avg.add(cost.item())

        if i % opt.valInterval == 0 and i != 0:
            t0 = time.time()
            vloss, vacc = validate(model, criterion, val_loader, converter, opt, iteration=i)
            model.train()
            if vacc > best_acc:
                best_acc = vacc
                torch.save(model.state_dict(), f'./saved_models/{opt.experiment_name}/best_accuracy.pth')
            print(f'[{i}/{opt.num_iter}] Loss: {loss_avg.val:.5f}, Val Loss: {vloss:.5f}, Val Acc: {vacc:.4f}, Best: {best_acc:.4f}, LR: {scheduler.get_last_lr()[0]:.6f}, Time: {time.time()-t0:.1f}s')
        i += 1

    torch.save(model.state_dict(), f'./saved_models/{opt.experiment_name}/final.pth')
    print(f"\nDone! Best accuracy: {best_acc:.4f}")

if __name__ == '__main__':
    main()

In [ ]:
import yaml, os
os.makedirs('config_files', exist_ok=True)
config = {
    'experiment_name': 'cryptogram_digits',
    'train_data': 'all_data/train',
    'valid_data': 'all_data/val',
    'manualSeed': 1111, 'workers': 0, 'batch_size': 32,
    'num_iter': 30000, 'valInterval': 500,
    'saved_model': '',
    'FT': False, 'optim': 'adam', 'lr': 0.001,
    'beta1': 0.9, 'rho': 0.95, 'eps': 1e-8, 'grad_clip': 5,
    'select_data': 'train', 'batch_ratio': '1',
    'total_data_usage_ratio': 1.0, 'batch_max_length': 2,
    'imgH': 64, 'imgW': 200, 'rgb': False,
    'contrast_adjust': False, 'sensitive': True, 'PAD': True,
    'data_filtering_off': False,
    'Transformation': 'None', 'FeatureExtraction': 'VGG',
    'SequenceModeling': 'BiLSTM', 'Prediction': 'CTC',
    'num_fiducial': 20, 'input_channel': 1,
    'output_channel': 256, 'hidden_size': 256,
    'decode': 'greedy', 'new_prediction': False,
    'freeze_FeatureFxtraction': False, 'freeze_SequenceModeling': False,
    'number': '0123456789', 'symbol': '', 'lang_char': '',
}
with open('config_files/cryptogram_digits_config.yaml', 'w') as f:
    yaml.dump(config, f)
print("Config saved!")

In [ ]:
!python train_digits.py

In [ ]:
from google.colab import files
import os
p = 'saved_models/cryptogram_digits/best_accuracy.pth'
if os.path.exists(p):
    files.download(p)
    print("Model downloaded!")
else:
    print("Model not found")